In [ ]:
# ── Clone repo from GitHub (run once per Colab session) ───────────────────
import os, sys
if not os.path.exists('/content/protosearch'):
    !git clone https://github.com/bjdraper/protosearch.git /content/protosearch
sys.path.insert(0, '/content/protosearch')
!pip install -q -e /content/protosearch

# 02 — Clustering
1. PCA → FAISS KNN graph → Leiden community detection
2. t-SNE 2D layout
3. Sub-clustering of focal clusters (optional)

In [ ]:
import os, sys
import numpy as np
from pathlib import Path

# ── CONFIG (edit here) ────────────────────────────────────────────────────────
PROJECT_ROOT   = '/content/drive/MyDrive/acyltransferase'
K_NEIGHBORS    = 25      # KNN graph connectivity
RESOLUTION     = 2.0     # Leiden resolution (higher = more clusters)
PCA_DIMS       = 50
TSNE_PERP      = 100
SUBCLUSTER     = 'LCLAT1'  # top-level cluster label to sub-cluster; set None to skip
SUB_RESOLUTION = 1.0
SUB_TSNE_PERP  = 30
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from acyltransferase.config import load_config
from acyltransferase.embed  import load_embeddings
cfg = load_config('config.yaml')

In [ ]:
# Load embeddings
embeddings, ids = load_embeddings(
    ('data/embeddings/hits.npy', 'data/embeddings/hits_ids.txt'),
)
ref_emb, ref_ids = load_embeddings(
    ('data/embeddings/ref.npy', 'data/embeddings/ref_ids.txt'),
)
print(f'Hits: {embeddings.shape}   Refs: {ref_emb.shape}')

In [ ]:
# Run Leiden clustering
from acyltransferase.cluster import run_clustering
import os

os.makedirs('results/clustering', exist_ok=True)

result = run_clustering(
    embeddings     = embeddings,
    ids            = ids,
    ref_embeddings = ref_emb,
    ref_ids        = ref_ids,
    ref_colours    = cfg.probe_colours(),
    ref_labels     = cfg.probe_labels(),
    k_neighbors    = K_NEIGHBORS,
    resolution     = RESOLUTION,
    pca_dims       = PCA_DIMS,
    tsne_perp      = TSNE_PERP,
    tsne_cache     = 'data/embeddings/tsne_cache.npy',
)

result.assignments.to_csv('results/clustering/cluster_assignments.tsv', sep='\t', index=False)
result.summary.to_csv('results/clustering/cluster_summary.tsv', sep='\t', index=False)
print(result.summary.to_string(index=False))

In [ ]:
# t-SNE plot
from acyltransferase.visualize import plot_tsne
from sklearn.decomposition import PCA

# project ref probes into the same t-SNE space (approximate via PCA only)
pca = PCA(n_components=PCA_DIMS).fit(embeddings)
ref_pca = pca.transform(ref_emb)

fig = plot_tsne(
    coords        = result.tsne_coords,
    labels        = result.assignments['label'].tolist(),
    label_colours = result.label_colours,
    title         = 'Leiden clustering (all sequences)',
)
fig.savefig('results/clustering/tsne_clusters.png', dpi=150)
fig.show()

In [ ]:
# Sub-clustering (optional)
if SUBCLUSTER:
    from acyltransferase.cluster import run_clustering
    from acyltransferase.utils   import read_fasta, write_fasta

    sub_ids = result.assignments.loc[
        result.assignments['label'] == SUBCLUSTER, 'protein_id'
    ].tolist()
    print(f'Sub-clustering {SUBCLUSTER}: {len(sub_ids)} sequences')

    sub_result = run_clustering(
        embeddings     = embeddings,
        ids            = ids,
        ref_embeddings = ref_emb,
        ref_ids        = ref_ids,
        ref_colours    = cfg.probe_colours(),
        ref_labels     = cfg.probe_labels(),
        k_neighbors    = K_NEIGHBORS,
        resolution     = SUB_RESOLUTION,
        pca_dims       = PCA_DIMS,
        tsne_perp      = SUB_TSNE_PERP,
        subset_ids     = sub_ids,
    )

    slug = SUBCLUSTER.lower().replace(' ', '_')
    sub_result.assignments.to_csv(
        f'results/clustering/{slug}_subcluster_assignments.tsv', sep='\t', index=False
    )
    sub_result.summary.to_csv(
        f'results/clustering/{slug}_subcluster_summary.tsv', sep='\t', index=False
    )

    fig2 = plot_tsne(
        coords        = sub_result.tsne_coords,
        labels        = sub_result.assignments['label'].tolist(),
        label_colours = sub_result.label_colours,
        title         = f'{SUBCLUSTER} sub-clusters',
    )
    fig2.savefig(f'results/clustering/{slug}_subcluster_tsne.png', dpi=150)
    fig2.show()

    # Export FASTA for tree building
    hits = read_fasta('data/hmmer_hits/proteins_filtered_hmmer.faa')
    hit_ids = set(sub_ids)
    sub_hits = [(rid, seq) for rid, seq in hits if rid in hit_ids]
    write_fasta(sub_hits, f'results/{slug}_hits.faa')
    print(f'Exported {len(sub_hits)} sequences → results/{slug}_hits.faa')

## ✓ Done
Next: **03_trees_asr.ipynb** — phylogenetics and ancestral reconstruction.